In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import re
import os

# Configurar opciones de Selenium
options = Options()
#options.add_argument("--headless")  # Puedes activar modo headless si lo deseas
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

def normalize_percent_str(s: str) -> str:
    """
    Normaliza strings tipo '-21%' / '21%' a un porcentaje positivo: '21%'.
    Si no parece porcentaje, regresa ''.
    """
    if not s:
        return ""
    s = str(s).strip()
    m = re.search(r'(-?\d+(?:[\.,]\d+)?)\s*%', s)
    if not m:
        return ""
    val = m.group(1).replace(",", ".")
    try:
        num = abs(float(val))
    except:
        return ""
    if num.is_integer():
        return f"{int(num)}%"
    return f"{num}%"

def extraer_info_producto_elektra(product, fecha_consulta):
    """
    Extrae información de cada producto de Elektra
    """
    try:
        # Nombre del producto - desde el span con clase productBrand
        nombre = ""
        nombre_tag = product.find("span", class_="vtex-product-summary-2-x-productBrand")
        if nombre_tag:
            nombre = nombre_tag.get_text(strip=True)
        
        # Marca (extraer del nombre) - EN MAYÚSCULAS
        marca = ""
        if nombre:
            palabras = nombre.split()
            if palabras:
                marca = palabras[0].upper()
        
        # Color (extraer del nombre del producto)
        color = ""
        if nombre:
            nombre_lower = nombre.lower()
            # Lista de colores comunes
            colores = [
                'titanio negro', 'titanio blanco', 'titanio del desierto', 'titanio natural', 'negro', 
                'black', 'blanco', 'white', 'azul', 'blue', 'rojo', 'red',
                'verde', 'green', 'amarillo', 'yellow', 'rosa', 'pink', 'morado', 'purple', 'púrpura',
                'gris', 'gray', 'grey', 'dorado', 'gold', 'plateado', 'silver', 'plata', 'cian',
                'lila', 'grafito', 'purpura', 'lavanda', 'ultramarino', 'medianoche', 'café',
                'menta', 'multicolor', 'beige','violeta', 'naranja', 'midnight', 'titanio', 'starlight'
                ]
            
            for c in colores:
                if c in nombre_lower:
                    color = c.capitalize()
                    break
        
        # Capacidad (extraer del nombre del producto) - CON ESPACIO
        capacidad = ""
        if nombre:
            # Buscar patrones de capacidad como: 64GB, 128GB, 256GB, 512GB, 1TB, etc.
            capacidad_match = re.search(r'(\d+(?:\.\d+)?)\s*(GB|TB)', nombre, re.IGNORECASE)
            if capacidad_match:
                numero = capacidad_match.group(1)
                unidad = capacidad_match.group(2).upper()
                capacidad = f"{numero} {unidad}"  # Agregar espacio entre número y unidad
        
        # URL del producto - desde el enlace principal
        url_producto = ""
        # Buscar en el elemento padre que contiene el enlace
        if hasattr(product, 'name') and product.name == 'a':
            # Si el product ya es el enlace
            if product.get("href"):
                href = product["href"]
                if href.startswith("/"):
                    url_producto = "https://www.elektra.mx" + href
                elif href.startswith("https://"):
                    url_producto = href
        else:
            # Buscar el enlace dentro del product
            link_tag = product.find("a", class_="vtex-product-summary-2-x-clearLink")
            if not link_tag:
                link_tag = product.find("a", href=True)
            if link_tag and link_tag.get("href"):
                href = link_tag["href"]
                if href.startswith("/"):
                    url_producto = "https://www.elektra.mx" + href
                elif href.startswith("https://"):
                    url_producto = href
        
        # SKU - extraer del atributo data-id en el elemento cite
        sku = ""
        cite_tag = product.find("cite", {"data-id": True})
        if cite_tag:
            sku = cite_tag.get("data-id", "")
        
        # Si no se encuentra en cite, extraer de la URL
        if not sku and url_producto:
            sku_match = re.search(r'-(\d{8})/p$', url_producto)
            if sku_match:
                sku = sku_match.group(1)
            else:
                # Patrón alternativo para otros formatos
                sku_match = re.search(r'/([^/]*-)?(\d{6,})/p$', url_producto)
                if sku_match:
                    sku = sku_match.group(2)
        
        # Precios
        precio_original = ""
        precio_descuento = ""
        porcentaje_descuento = ""
        
        def _first_text(node):
            if not node:
                return ""
            return node.get_text(strip=True)
        
        # --- NUEVO CONTENEDOR DE PRECIOS (PLP) ---
        # Con descuento:
        #   div.listPrice span  (precio original)
        #   div.containerPricePlp div.price span (precio oferta)
        #   div.containerPricePlp div.descuento (ej -21%)
        # Sin descuento:
        #   div.containerPricePlp div.priceNoList span (precio actual)
        list_price_new = product.select_one("div.listPrice span")
        selling_price_new = product.select_one("div.containerPricePlp div.price span")
        no_list_price_new = product.select_one("div.containerPricePlp div.priceNoList span")
        descuento_new = product.select_one("div.containerPricePlp div.descuento")
        
        if list_price_new or selling_price_new or no_list_price_new:
            # Prioridad: si hay price (oferta) úsalo; si no, usa priceNoList (sin descuento)
            precio_original = _first_text(list_price_new)
            precio_descuento = _first_text(selling_price_new) or _first_text(no_list_price_new)
            porcentaje_descuento = normalize_percent_str(_first_text(descuento_new))
        
        # Si no hay listPrice y el precio viene en priceNoList, entonces no hay descuento real:
        # dejar ambos (original y descuento) como el mismo precio y marcar No discount
        if no_list_price_new and not list_price_new:
            raw = _first_text(no_list_price_new)
            if raw:
                precio_original = raw
                precio_descuento = "No discount"
                porcentaje_descuento = ""
        
        # --- SELECTORES ANTERIORES (fallback) ---
        if not precio_original and not precio_descuento:
            # Precio original (tachado) - listPrice
            precio_orig_tag = product.find("span", class_="vtex-store-components-3-x-listPriceValue")
            if precio_orig_tag:
                # Buscar el span con currencyContainer
                precio_span = precio_orig_tag.find("span", class_="vtex-store-components-3-x-currencyContainer")
                if precio_span:
                    # Buscar el span interno que contiene el precio
                    precio_inner_span = precio_span.find("span")
                    if precio_inner_span:
                        precio_original = precio_inner_span.get_text(strip=True)
            
            # Precio con descuento - sellingPrice
            precio_desc_tag = product.find("span", class_="vtex-store-components-3-x-sellingPriceValue")
            if precio_desc_tag:
                # Buscar el span interno que contiene el precio
                precio_span = precio_desc_tag.find("span")
                if precio_span:
                    precio_descuento = precio_span.get_text(strip=True)
            
            # Si no se encuentra, buscar en la estructura del div sellingPrice
            if not precio_descuento:
                precio_desc_div = product.find("div", class_="vtex-store-components-3-x-sellingPrice")
                if precio_desc_div:
                    precio_span = precio_desc_div.find("span", class_="vtex-store-components-3-x-currencyContainer")
                    if precio_span:
                        precio_inner_span = precio_span.find("span")
                        if precio_inner_span:
                            precio_descuento = precio_inner_span.get_text(strip=True)
            
            # Si aún no se encuentra, buscar directamente el currencyContainer
            if not precio_descuento:
                precio_desc_tag = product.find("span", class_="vtex-store-components-3-x-currencyContainer")
                if precio_desc_tag:
                    precio_span = precio_desc_tag.find("span")
                    if precio_span:
                        precio_descuento = precio_span.get_text(strip=True)
        
        # Lógica de precios corregida
        if precio_original and precio_descuento:
            pass
        elif not precio_original and precio_descuento:
            precio_original = precio_descuento
            precio_descuento = "No discount"
        elif precio_original and not precio_descuento:
            precio_descuento = "No discount"
        else:
            precio_original = "No inventory"
            precio_descuento = "No discount"
        
        # Calcular porcentaje de descuento
        def limpiar_precio(precio):
            if precio and precio not in ["No discount", "No inventory"]:
                cleaned = re.sub(r'[^\d.]', '', precio.replace(',', ''))
                try:
                    return float(cleaned)
                except:
                    return None
            return None
        
        p_orig = limpiar_precio(precio_original)
        p_desc = limpiar_precio(precio_descuento)
        
        # Porcentaje final: 1) usar el del sitio (normalizado, positivo) si viene, 2) si no, calcular
        if porcentaje_descuento:
            porcentaje_str = porcentaje_descuento
        elif p_orig and p_desc and p_orig > p_desc:
            porcentaje_calc = abs(round(100 * (p_orig - p_desc) / p_orig, 2))
            porcentaje_str = f"{porcentaje_calc}%"
        else:
            porcentaje_str = "No discount"
        
        return {
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "SKU": sku,
            "URL": url_producto,
            "FechaConsulta": fecha_consulta,
            "Precio_original": precio_original,
            "Precio_descuento": precio_descuento,
            "Porcentaje_descuento": porcentaje_str
        }
        
    except Exception as e:
        print(f"Error al extraer producto: {e}")
        return None

def scrape_elektra_smartphones():
    """
    Función principal para hacer scraping de smartphones en Elektra
    """
    url = "https://www.elektra.mx/telefonia/celulares/smartphones?initialMap=c,c,c&initialQuery=telefonia/celulares/smartphones&map=category-1,category-2,category-3,sellername&query=/telefonia/celulares/smartphones/elektra&searchState"
    
    print("Iniciando scraping de Elektra...")
    
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(10)
    
    data = []
    fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        # Hacer scroll y buscar botón "Mostrar más" específico de Elektra
        max_clicks = 100  # Aumentar límite para cargar todos los productos
        clicks_realizados = 0
        
        while clicks_realizados < max_clicks:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(5)
            
            try:
                mostrar_mas_btn = driver.find_element(By.CSS_SELECTOR, "div.elektra-search-result-3-x-buttonShowMore button")
                if mostrar_mas_btn.is_displayed() and mostrar_mas_btn.is_enabled():
                    print(f"Haciendo clic en 'Mostrar más' (clic #{clicks_realizados + 1})")
                    driver.execute_script("arguments[0].scrollIntoView(true);", mostrar_mas_btn)
                    time.sleep(1)
                    driver.execute_script("arguments[0].click();", mostrar_mas_btn)
                    clicks_realizados += 1
                    time.sleep(15)
                else:
                    print("Botón 'Mostrar más' no está disponible (deshabilitado o no visible)")
                    break
            except Exception as e:
                print(f"No se encontró el botón 'Mostrar más' - Se han cargado todos los productos")
                break
        
        print(f"Total de clics en 'Mostrar más': {clicks_realizados}")
        print("Finalizando carga de productos...")
        
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3)
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        products = []
        products = soup.find_all("a", class_="vtex-product-summary-2-x-clearLink")
        print(f"Estrategia 1 - Enlaces encontrados: {len(products)}")
        
        if not products:
            products = soup.find_all("div", class_="vtex-search-result-3-x-galleryItem")
            print(f"Estrategia 2 - Elementos galleryItem: {len(products)}")
        
        if not products:
            products = soup.find_all("div", class_="vtex-product-summary-2-x-container")
            print(f"Estrategia 3 - Contenedores de producto: {len(products)}")
        
        if not products:
            products = soup.find_all("article", class_="vtex-product-summary-2-x-element")
            print(f"Estrategia 4 - Artículos de producto: {len(products)}")
        
        print(f"Productos encontrados: {len(products)}")
        
        # Debug: imprimir información de los primeros productos
        for i, product in enumerate(products[:3]):
            print(f"\n--- Debug Producto {i+1} ---")
            print(f"Tipo de elemento: {product.name}")
            print(f"Clases: {product.get('class', [])}")
            
            new_list = product.select_one("div.listPrice span")
            new_sell = product.select_one("div.containerPricePlp div.price span")
            new_no_list = product.select_one("div.containerPricePlp div.priceNoList span")
            new_desc = product.select_one("div.containerPricePlp div.descuento")
            print(f"Nuevo layout listPrice encontrado: {bool(new_list)}")
            print(f"Nuevo layout price encontrado: {bool(new_sell)}")
            print(f"Nuevo layout priceNoList encontrado: {bool(new_no_list)}")
            print(f"Nuevo layout descuento encontrado: {bool(new_desc)}")
            if new_list:
                print(f"Nuevo precio original: {new_list.get_text(strip=True)}")
            if new_sell:
                print(f"Nuevo precio descuento: {new_sell.get_text(strip=True)}")
            if new_no_list:
                print(f"Nuevo precio sin descuento (priceNoList): {new_no_list.get_text(strip=True)}")
            if new_desc:
                print(f"Nuevo % descuento (raw): {new_desc.get_text(strip=True)}")
                print(f"Nuevo % descuento (normalizado): {normalize_percent_str(new_desc.get_text(strip=True))}")
        
        for product in products:
            info = extraer_info_producto_elektra(product, fecha_consulta)
            if info and (info["Nombre"] or info["URL"]):
                data.append(info)
                print(f"Producto extraído: {info['Nombre'][:30] if info['Nombre'] else 'Sin nombre'} - Color: {info['Color']} - Capacidad: {info['Capacidad']} - SKU: {info['SKU']} - URL: {info['URL'][:50] if info['URL'] else 'Sin URL'}")
        
        print(f"Productos válidos extraídos: {len(data)}")
        
    except Exception as e:
        print(f"Error durante el scraping: {e}")
        import traceback
        traceback.print_exc()
    finally:
        driver.quit()
    
    return data

# Ejecutar el scraping
resultados_elektra = scrape_elektra_smartphones()

# Exportar a CSV
if resultados_elektra:
    df_elektra = pd.DataFrame(resultados_elektra)
    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"elektra_smartphones_{fecha_hoy}.csv"
    try:
        df_elektra.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
    except PermissionError:
        # En Windows es común que el archivo esté abierto/bloqueado (Excel, preview, etc.)
        sufijo = datetime.now().strftime("%H%M%S")
        nombre_archivo_alt = f"elektra_smartphones_{fecha_hoy}_{sufijo}.csv"
        df_elektra.to_csv(nombre_archivo_alt, index=False, encoding="utf-8-sig")
        nombre_archivo = nombre_archivo_alt
    print(f"Datos exportados a {nombre_archivo}")
    print(f"Total de productos: {len(df_elektra)}")
    print("\nPrimeros 5 productos:")
    print(df_elektra.head())
else:
    print("No se encontraron productos.")

Iniciando scraping de Elektra...
Haciendo clic en 'Mostrar más' (clic #1)
Haciendo clic en 'Mostrar más' (clic #2)
Haciendo clic en 'Mostrar más' (clic #3)
Haciendo clic en 'Mostrar más' (clic #4)
Haciendo clic en 'Mostrar más' (clic #5)
Haciendo clic en 'Mostrar más' (clic #6)
Haciendo clic en 'Mostrar más' (clic #7)
No se encontró el botón 'Mostrar más' - Se han cargado todos los productos
Total de clics en 'Mostrar más': 7
Finalizando carga de productos...
Estrategia 1 - Enlaces encontrados: 448
Productos encontrados: 448

--- Debug Producto 1 ---
Tipo de elemento: a
Clases: ['vtex-product-summary-2-x-clearLink', 'vtex-product-summary-2-x-clearLink--customPricePlp', 'vtex-product-summary-2-x-clearLink--OutletProductsNotAvailable', 'h-100', 'flex', 'flex-column']
Nuevo layout listPrice encontrado: True
Nuevo layout price encontrado: True
Nuevo layout priceNoList encontrado: False
Nuevo layout descuento encontrado: True
Nuevo precio original: $9,999
Nuevo precio descuento: $8,799
Nuev